# Predicting Loan Default: A Machine Learning Approach to Credit Risk Assessment

**Author:** Atilla Ahmed
**Dataset:** Lending Club Loan Data (2007–2018), ~2.26M loans, 145 features

---

## Table of Contents
1. [Setup and Imports](#1-setup-and-imports)
2. [Introduction and Problem Formulation](#2-introduction-and-problem-formulation)
3. [Data Loading and Initial Inspection](#3-data-loading-and-initial-inspection)
4. [Data Cleaning and Preprocessing](#4-data-cleaning-and-preprocessing)
5. [Exploratory Data Analysis](#5-exploratory-data-analysis)
6. [Feature Engineering & Preprocessing Pipeline](#6-feature-engineering)
7. [Mathematical Framework](#7-mathematical-framework)
8. [Baseline Models](#8-baseline-models)
9. [Advanced Models and Hyperparameter Tuning](#9-advanced-models-and-hyperparameter-tuning)
10. [Model Evaluation and Comparison](#10-model-evaluation-and-comparison)
11. [Business Impact Analysis](#11-business-impact-analysis)
12. [Conclusions, Limitations, and Future Work](#12-conclusions-limitations-and-future-work)
13. [References](#13-references)

## 1. Setup and Imports <a id="1-setup-and-imports"></a>

All libraries and global settings are defined in one place so the notebook is fully reproducible. The random seed is fixed at 42 throughout.

In [2]:
# SETUP — libraries, global config, random seed

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, cross_val_score)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              confusion_matrix, classification_report,
                              roc_curve, precision_recall_curve)
from scipy.stats import randint, uniform

SEED = 42
np.random.seed(SEED)


## 2. Introduction and Problem Formulation <a id="2-introduction-and-problem-formulation"></a>

### 2.1 Background

Lending Club was the largest peer-to-peer lending platform in the US. Investors lend money directly to borrowers, and the central risk they face is credit default a borrower stops repaying. When that happens the investor loses part or all of their principal. Predicting which borrowers will default is therefore critical for anyone deploying capital on the platform.

This project builds a classification model to predict loan default using only information available at the time of loan issuance. No post-origination data is used.

### 2.2 Problem Statement

**Task:** Binary classification - *Fully Paid* (0) vs *Charged Off / Default* (1).

**Target:** `loan_status`, filtered to loans with a known final outcome only.

**Key constraint:** Only features available at origination. Any variable that reflects what happened during repayment is excluded using those would be data leakage and would produce a model that works perfectly in training but fails entirely on new applications.

**Evaluation metric:** Accuracy is misleading on imbalanced datasets. Missing a default costs real money; rejecting a good borrower costs opportunity. This asymmetry makes Recall and PR-AUC far more relevant.

### 2.3 Expected Loss Framework

$$\text{Expected Loss} = PD \times LGD \times EAD$$

- **PD (Probability of Default)** - the likelihood a borrower fails to repay. This is what the model predicts.
- **LGD (Loss Given Default)** - the fraction of the outstanding amount that is not recovered. Typically 40–60% for unsecured consumer loans.
- **EAD (Exposure at Default)** - the total amount owed at the time of default.

The model targets the PD component. A well-calibrated PD estimate allows a lender to price risk correctly and make informed accept/reject decisions on new applications.

## 3. Data Loading and Initial Inspection <a id="3-data-loading-and-initial-inspection"></a>

The dataset is a single 1.1 GB CSV file with 2.26 million loan records and 145 columns. A data dictionary is included as a separate Excel file.